# Name That LLM – LinearSVC auf Open Pool + train.csv

Trainiert auf dem externen Datensatz *Name That LLM Open Pool* (crtproduction, CC BY-SA 4.0) plus `train.csv`, Features: Wort- + Char-TF-IDF + Stilometrie. Ausgabe: `submission.parquet`. Kein Internet nötig.

**Benötigte Inputs:** Competition `name-that-llm` und Dataset `crtproduction/name-that-llm-open-pool`.

In [ ]:
# Parameter (aus experiments/06_external_data/run_fast.py)
PER_CLASS = None      # Pool-Texte je Modell (None = kompletter Pool)
REAL_WEIGHT = 5.0  # Gewicht der echten train.csv-Texte
C = 0.1
KIMI_HIGH_ONLY = False
SEED = 42

In [ ]:
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd

def find(pattern, env, fallback):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    return Path(hits[0]) if hits else Path(fallback)

KAGGLE_DIR = Path("/kaggle/input/competitions/name-that-llm")
if KAGGLE_DIR.exists():
    DATA_DIR = KAGGLE_DIR
elif os.environ.get("NTL_DATA_DIR"):
    DATA_DIR = Path(os.environ["NTL_DATA_DIR"])
else:
    DATA_DIR = find("train.csv", "_UNUSED", "../../data/train.csv").parent
POOL_PATH = find("ntl_open_pool.parquet", "NTL_POOL", "../../data/external/open_pool/ntl_open_pool.parquet")
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
print("DATA_DIR:", DATA_DIR, "| POOL:", POOL_PATH, "| OUT_DIR:", OUT_DIR)
assert (DATA_DIR / "train.csv").exists() and POOL_PATH.exists(), "Input fehlt – Open-Pool-Dataset eingebunden?"

## Features & Modell (aus `src/features.py` und `src/linear_large.py`)

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import StandardScaler

# Zeichen, deren Häufigkeit (pro 100 Wörter) als Feature dient
CHAR_COUNTS = {
    "em_dash": "—",
    "en_dash": "–",
    "hyphen": "-",
    "curly_apos": "’",
    "curly_dquote": "“",
    "straight_apos": "'",
    "straight_dquote": '"',
    "colon": ":",
    "semicolon": ";",
    "comma": ",",
    "paren": "(",
    "question": "?",
    "exclam": "!",
    "ellipsis": "…",
}

_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
_WORD = re.compile(r"\b\w+\b")


def stylometric_features(texts: pd.Series) -> pd.DataFrame:
    """Längen-, Satz-, Wort- und Interpunktionsstatistiken je Text."""
    rows = []
    for text in texts:
        words = _WORD.findall(text)
        n_words = max(len(words), 1)
        sents = [s for s in _SENT_SPLIT.split(text.strip()) if s]
        sent_lens = [len(_WORD.findall(s)) for s in sents] or [0]
        row = {
            "n_chars": len(text),
            "n_words": len(words),
            "n_sents": len(sents),
            "n_paras": text.count("\n\n") + 1,
            "words_per_sent": np.mean(sent_lens),
            "sent_len_std": np.std(sent_lens),
            "avg_word_len": np.mean([len(w) for w in words]) if words else 0,
            "type_token_ratio": len({w.lower() for w in words}) / n_words,
            "upper_ratio": sum(c.isupper() for c in text) / max(len(text), 1),
            "non_ascii_ratio": sum(ord(c) > 127 for c in text) / max(len(text), 1),
            "digit_ratio": sum(c.isdigit() for c in text) / max(len(text), 1),
            "ends_with_period": float(text.rstrip().endswith(".")),
        }
        for name, ch in CHAR_COUNTS.items():
            row[f"per100w_{name}"] = 100 * text.count(ch) / n_words
        rows.append(row)
    return pd.DataFrame(rows, index=texts.index)


class StyloTransformer(BaseEstimator, TransformerMixin):
    """sklearn-Wrapper um stylometric_features (für Pipelines)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return stylometric_features(pd.Series(X)).to_numpy(dtype=float)


def word_tfidf(**kw) -> TfidfVectorizer:
    params = dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=False)
    return TfidfVectorizer(**{**params, **kw})


def char_tfidf(**kw) -> TfidfVectorizer:
    params = dict(analyzer="char", ngram_range=(1, 5), min_df=3, sublinear_tf=True, lowercase=False)
    return TfidfVectorizer(**{**params, **kw})


def text_features(use_stylo: bool = True):
    """FeatureUnion aus Wort-TF-IDF, Char-TF-IDF und (optional) skalierten Stilo-Features."""
    parts = [word_tfidf(), char_tfidf()]
    if use_stylo:
        parts.append(make_pipeline(StyloTransformer(), StandardScaler()))
    return make_union(*parts)

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.special import softmax
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


STYLO_WEIGHT = 0.1
SVM_TEMPERATURE = 5.0  # Skalierung der SVM-Scores für Pseudo-Wahrscheinlichkeiten (softmax)


class LargeVectorizer:
    """Wort- + Char-TF-IDF (begrenztes Vokabular, float32) + gewichtete Stilo-Features."""

    def __init__(self, max_word: int = 300_000, max_char: int = 500_000,
                 stylo_weight: float = STYLO_WEIGHT):
        self.word = word_tfidf(min_df=3, max_features=max_word, dtype=np.float32)
        self.char = char_tfidf(min_df=5, max_features=max_char, dtype=np.float32)
        self.scaler = StandardScaler()
        self.stylo_weight = stylo_weight

    def _stack(self, texts, fit: bool):
        if fit:
            w, c = self.word.fit_transform(texts), self.char.fit_transform(texts)
            s = self.scaler.fit_transform(stylometric_features(texts))
        else:
            w, c = self.word.transform(texts), self.char.transform(texts)
            s = self.scaler.transform(stylometric_features(texts))
        parts = [w, c]
        if self.stylo_weight:
            parts.append(sp.csr_matrix(s * self.stylo_weight))
        return sp.hstack(parts, format="csr", dtype=np.float32)

    def fit_transform(self, texts):
        return self._stack(texts, fit=True)

    def transform(self, texts):
        return self._stack(texts, fit=False)


def make_svm(C: float = 0.1):
    return LinearSVC(C=C, class_weight="balanced")


def svm_proba(clf, X, temperature: float = SVM_TEMPERATURE) -> np.ndarray:
    """Softmax über die SVM-Scores (für Ensembles; Temperatur später auf OOF kalibrieren)."""
    return softmax(clf.decision_function(X) * temperature, axis=1)


def sample_weights(n_pool: int, n_real: int, real_weight: float) -> np.ndarray:
    """Gewicht je Zeile: Pool = 1, echte train.csv-Texte = real_weight."""
    return np.concatenate([np.ones(n_pool), np.full(n_real, real_weight)]).astype(np.float32)

## Daten

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

pool = pd.read_parquet(POOL_PATH, columns=["text", "model", "gen_reasoning"])
if KIMI_HIGH_ONLY:
    pool = pool[(pool["model"] != "kimi-k3") | (pool["gen_reasoning"] == "high")]
if PER_CLASS:
    pool = pool.sample(frac=1, random_state=SEED).groupby("model").head(PER_CLASS)
assert set(pool["model"]) == set(train["model"])
print("train", train.shape, "| pool", pool.shape, "| test", test.shape)

## 5-Fold-OOF auf train.csv (Pool + train-Folds)

Nacheinander statt parallel: Ein Fit braucht ~14 GB RAM (liblinear kopiert die Matrix).

In [ ]:
import time
import scipy.sparse as sp

FOLD_OF_PROMPT = {"p081": 3, "p073": 1, "p120": 4, "p097": 2, "p065": 4, "p090": 4, "p036": 3, "p011": 3, "p139": 0, "p106": 3, "p003": 1, "p019": 0, "p062": 2, "p142": 2, "p108": 1, "p047": 2, "p022": 2, "p101": 3, "p046": 3, "p018": 1, "p099": 0, "p134": 0, "p051": 3, "p082": 2, "p013": 1, "p014": 0, "p023": 1, "p068": 1, "p043": 1, "p141": 3, "p131": 3, "p049": 0, "p143": 1, "p001": 2, "p008": 1, "p089": 0, "p015": 4, "p024": 0, "p083": 1, "p148": 1, "p118": 1, "p095": 4, "p102": 2, "p146": 3, "p130": 4, "p009": 0, "p091": 3, "p128": 1, "p012": 2, "p053": 1, "p137": 2, "p144": 0, "p040": 4, "p039": 0, "p140": 4, "p028": 1, "p132": 2, "p145": 4, "p086": 3, "p116": 3, "p056": 3, "p070": 4, "p006": 3, "p093": 1, "p117": 2, "p071": 3, "p126": 3, "p025": 4, "p035": 4, "p136": 3, "p030": 4, "p109": 0, "p124": 0, "p059": 0, "p077": 2, "p100": 4, "p138": 1, "p063": 1, "p114": 0, "p113": 1, "p007": 2, "p107": 2, "p052": 2, "p087": 2, "p042": 2, "p060": 4, "p002": 4, "p103": 1, "p045": 4, "p104": 0, "p038": 1, "p123": 1, "p069": 0, "p004": 0, "p074": 0, "p017": 2, "p110": 4, "p133": 1, "p033": 1, "p037": 2, "p129": 0, "p067": 2, "p054": 0, "p079": 0, "p076": 3, "p135": 4, "p021": 3, "p115": 4, "p147": 2, "p119": 0, "p061": 3, "p094": 0, "p127": 2, "p088": 1, "p044": 0, "p072": 2, "p005": 4, "p016": 3, "p000": 3, "p057": 2, "p096": 3, "p080": 4, "p125": 4, "p048": 1, "p032": 2, "p026": 3, "p075": 4, "p105": 4, "p020": 4, "p055": 4, "p027": 2, "p111": 3, "p041": 3, "p092": 2, "p010": 4, "p084": 0, "p064": 0, "p034": 0, "p085": 4, "p098": 1, "p066": 3, "p112": 2, "p121": 3, "p050": 4, "p078": 1, "p031": 3, "p029": 0, "p149": 0, "p058": 1, "p122": 2}
folds = train["prompt_id"].map(FOLD_OF_PROMPT).values

t0 = time.time()
vec = LargeVectorizer()
X_pool = vec.fit_transform(pool["text"])
X_train = vec.transform(train["text"])
y_pool, y_train = pool["model"].values, train["model"].values
print(f"Vektorisierung {(time.time() - t0) / 60:.1f} min, {X_pool.shape[1]} Merkmale", flush=True)

oof = None
for k in range(5):
    tr, va = folds != k, folds == k
    X = sp.vstack([X_pool, X_train[tr]], format="csr")
    y = np.concatenate([y_pool, y_train[tr]])
    clf = make_svm(C).fit(X, y, sample_weight=sample_weights(len(y_pool), tr.sum(), REAL_WEIGHT))
    del X
    if oof is None:
        oof = np.zeros((len(train), len(clf.classes_)), dtype=np.float32)
    oof[va] = clf.decision_function(X_train[va])
    acc = (clf.classes_[oof[va].argmax(1)] == y_train[va]).mean()
    print(f"Fold {k} fertig: acc={acc:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
    out = pd.DataFrame(oof, columns=clf.classes_)
    out.insert(0, "id", train["id"].values)
    out.to_parquet(OUT_DIR / "svm_oof_decision.parquet", index=False)   # nach jedem Fold sichern

In [ ]:
from sklearn.metrics import balanced_accuracy_score
pred = clf.classes_[oof.argmax(1)]
print("OOF Balanced Accuracy:", round(balanced_accuracy_score(y_train, pred), 4))